# Ch.1　データを「読む」── 【講師用完全版】

| 項目 | 内容 |
|------|------|
| **使用データ** | ワインデータ（`load_wine()`）178件・13特徴量・3品種 |
| **作るもの** | 品種別・主要指標の集計表 |
| **実務での対応物** | 月次売上レポート / KPI ダッシュボードの集計表 |
| **演習時間** | 40分（問3まで完了で合格） |

---

> 📌 **講師メモ：このChの最大目的**  
> 「データをもらったら最初に何をするか」の習慣を植え付けること。  
> コードの書き方より「head → info → describe → isnull の4点確認」の習慣が大事。  
> 演習後に「なぜこの順番でやるのか」を口頭で振り返ると定着する。

---
## 🤖 ブラウザ版AIの使い方

JupyterLab を左画面・AIを右画面に並べて作業してください。

| ツール | URL |
|--------|-----|
| Microsoft Copilot | https://copilot.microsoft.com |
| ChatGPT | https://chat.openai.com |

> 📄 プロンプト集 → `ch1_prompt_guide.md`

---
## 🔧 準備 ── ライブラリの読み込み

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine

print("✅ ライブラリの読み込み完了")

---
## データを読み込む

> 📌 **講師メモ：**  
> 「実務では `pd.read_csv()` を使う」と一言添えて、  
> 「今日はネット不要の内蔵データを使う」と説明する（環境依存のトラブル防止）。

In [ ]:
wine = load_wine()
df   = pd.DataFrame(wine.data, columns=wine.feature_names)
df["品種"]  = wine.target
df["品種名"] = df["品種"].map({0: "Barolo", 1: "Grignolino", 2: "Barbera"})

print(f"✅ 読み込み完了  行数: {len(df)}  列数: {len(df.columns)}")

---
## 📋 データの「形」を確認する

> 📌 **講師メモ：**  
> 「新しいデータをもらったら最初にやること」として強調する。  
> 「まず地図を広げる」というアナロジーが有効。  
> 各コマンドは「答える問い」と対応させて説明する：  
> - `head()` → 「何が入っているか」  
> - `info()` → 「型と欠損はどうか」  
> - `describe()` → 「値の大きさと散らばりは？」  
> - `isnull().sum()` → 「空欄はないか」

### STEP 1｜先頭5行を目で確認する（`head()`）

In [ ]:
df.head()

### STEP 2｜型・件数・欠損値を確認する（`info()`）

確認ポイント：`Non-Null Count` が 178 と一致していれば欠損なし

In [ ]:
df.info()

### STEP 3｜基本統計量を確認する（`describe()`）

> 📌 **講師メモ（発問）：**  
> 「std が大きい列とはどういう意味ですか？」  
> → 「値がばらついている = 品種間の差が大きい = 分類に効く変数の候補」につなげる。

In [ ]:
df.describe().round(2)

### STEP 4｜欠損値を数える（`isnull().sum()`）

In [ ]:
missing = df.isnull().sum()
print(missing)
print(f"\n欠損値の合計: {missing.sum()} 件  ← 0 なら問題なし")

---
## 問1｜品種ごとのデータ件数を確認する ★☆☆

**実務での対応：** 「各商品カテゴリが何件あるか」を確認する作業と同じ。

> 📌 **講師メモ：**  
> 「件数の偏り（不均衡データ）がなぜ問題か」を Ch.5 と接続して予告する。  
> 「乳がんデータでは悪性37%・良性63%の偏りがある」と伏線を張る。  
> 時間目安：5分以内で完了するはず。

In [ ]:
# 品種番号ごとの件数
df["品種"].value_counts()

In [ ]:
# 品種名ごとの件数（番号より読みやすい）
df["品種名"].value_counts()

#### ✅ 解答のポイント
- Barolo: 59件、Grignolino: 71件、Barbera: 48件
- 合計178件
- 最大と最小の差は約1.5倍 → 偏りは許容範囲

---
## 問2｜品種別の集計表を作る ★★☆

> 📌 **講師メモ：**  
> Excel のピボットテーブルと対比して説明すると理解が早い。  
> `groupby().mean()` → `groupby()[複数列].mean()` → `.agg()` の順で段階的に広げる。  
> 「agg の結果を見て、どの品種が何で特徴的か」を声に出して読む練習をさせる。

### 1列の集計（alcohol のみ）

In [ ]:
df.groupby("品種名")["alcohol"].mean().round(2)

### 複数列を一度に集計

In [ ]:
result = df.groupby("品種名")[["alcohol", "proline"]].mean().round(2)
result

### 平均・最大・最小・標準偏差をまとめて出す

In [ ]:
df.groupby("品種名")[["alcohol", "proline"]].agg(["mean", "max", "min", "std"]).round(2)

#### ✅ 解答のポイント（講師の解説用）

| 品種 | alcohol 特徴 | proline 特徴 |
|------|-------------|-------------|
| Barolo | 最も高い（平均 13.7） | 最も高い（平均 1116） |
| Grignolino | 中間（平均 12.3） | 低い（平均 519） |
| Barbera | 最も低い（平均 13.1） | 中間（平均 625） |

→ 「Barolo はアルコールとプロリンの両方で他品種と差がある」が最大の気づき。  
→ Ch.3 の特徴量重要度で `proline` が上位に来ることの伏線になる。

---
## 問3｜集計結果から考察する ★★☆（最重要）

> 📌 **講師メモ：**  
> 考察は採点しない。「正解はない」と明示して心理的安全性を確保する。  
> 発表を求める場合は「気になった数字を一つ選んで一言だけ」でOK。  
> Ch.3 の特徴量重要度との「答え合わせ」として機能させることが最大のポイント。

> 💡 **想定される考察例：**  
> 「Barolo はプロリンが突出して高く、他の2品種の約2倍ある。土壌や気候の違いが関係しているかもしれない。Ch.3 ではプロリンが重要な変数になると予想する。」

### 考察欄

**気づいた品種・特徴量：**  

**なぜそうなると思いますか？**  

**次の分類モデルで効くと思う変数は？**

---
## 問4（発展）｜条件でデータを絞り込む ★★★

> 📌 **講師メモ：**  
> 「Excel のフィルター機能と同じことをコードで書く」と伝えると直感的に理解される。  
> AND条件の括弧必須ルールは必ずつまずくポイント。エラーが出たら全体でフォローする。

### 1条件フィルタ

In [ ]:
high_alc = df[df["alcohol"] >= 13.5]
print(f"条件を満たすワイン: {len(high_alc)} 件  （全体の {len(high_alc)/len(df):.0%}）")
high_alc[["品種名", "alcohol", "proline"]].head(10)

### 2条件フィルタ（AND 条件）

> **重要：** `&` でつなぎ、各条件を `()` で囲む（括弧がないと `ValueError`）

In [ ]:
premium = df[(df["alcohol"] >= 13.5) & (df["proline"] >= 700)]
print(f"両方の条件を満たすワイン: {len(premium)} 件")
premium["品種名"].value_counts()

### 絞り込んだデータでさらに集計する

In [ ]:
premium.groupby("品種名")[["alcohol", "proline"]].mean().round(2)

#### ✅ 解答のポイント
- alcohol >= 13.5: 約60件
- alcohol >= 13.5 かつ proline >= 700: 約50件
- 絞り込み後はほぼ Barolo のみが残る → 「Barolo の特徴がより鮮明になる」